> **Superseded again (2026-09-25): use `--render-location colab`** (or the UI's "Xuất MP4 trên Colab"). It drives Colab account 1's TPU v6e-1 automatically, moves kits and results through private R2 presigned URLs and always releases the runtime — see `docs/colab-render.md`.
>
> ⚠️ This manual notebook uploads the kit (client footage included) and the result to **litterbox.catbox.moe, a public anonymous file host**. Do not use it for client work.

# Remotion Render Pipeline — MONA auto-editor

> **Superseded by the automated cloud-render path** — `--render-location cloud` in
> `lib/talking_head_edit/cli.py` now rents, renders, downloads and destroys a Vast.ai box for you
> (announce + approval, hard cost ceilings, guaranteed teardown, no manual `curl`/copy-paste steps).
> See `docs/cloud-render.md` for setup and usage. **This notebook is kept as the free/manual
> alternative** — Colab's free-tier high-CPU runtime costs nothing, and the benchmark below (44-core
> EPYC, 93 s 1080p in ~2m51s) is the calibration source for `render_seconds_per_video_second` in
> `config/cloud-render.json`. Use this notebook when you want the Colab machine specifically (e.g.
> no Vast.ai account) rather than a rented box.

Render video Remotion trên Colab high-CPU (đã benchmark: 44-core EPYC render 1080p/93s trong ~2m51s, gấp ~2x local 12-core).

**Quy trình mỗi phiên:**
1. Đóng gói kit ở máy local (trong `remotion-composer/`):
   `tar czf mona_render_kit.tar.gz --force-local package.json package-lock.json tsconfig.json src public/mona_timeline_src.mp4 public/mona_timeline_props.json public/*.mp3`
2. Upload lấy link tự hủy 1h:
   `curl -s -F "reqtype=fileupload" -F "time=1h" -F "fileToUpload=@mona_render_kit.tar.gz" https://litterbox.catbox.moe/resources/internals/api.php`
3. Dán link vào `KIT_URL` ở cell SETUP → chạy lần lượt các cell.
4. Cell EXPORT trả link MP4 (tự hủy 1h) → tải về local.

**Ghi chú đã kiểm chứng:** single-process nhanh hơn chia chunk trên 1 máy (171s vs 313s); concurrency động `nproc-2`; NVENC không giúp (nghẽn ở vẽ frame, không phải encode).

In [ ]:
import os, subprocess, shutil
print("CPU cores:", os.cpu_count())
print(subprocess.run(["grep","-m1","model name","/proc/cpuinfo"],capture_output=True,text=True).stdout.strip())
print(subprocess.run(["free","-h"],capture_output=True,text=True).stdout.splitlines()[1])
try:
    gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],capture_output=True,text=True).stdout.strip()
except FileNotFoundError:
    gpu = "none (CPU runtime)"
print("GPU:", gpu or "none (CPU runtime)")
print("disk free:", shutil.disk_usage("/").free//2**30, "GB")

In [ ]:
%%bash
# [SETUP 1/2] Node 20 + tải project kit (chạy 1 lần mỗi phiên)
# >>> THAY KIT_URL bằng link litterbox mới (link cũ tự hủy sau 1h) <<<
KIT_URL="<DÁN_LINK_KIT_VÀO_ĐÂY>"
set -e
if ! command -v node >/dev/null || [[ "$(node -v)" != v2* ]]; then
  curl -sL https://deb.nodesource.com/setup_20.x | bash - > /tmp/node_setup.log 2>&1
  apt-get install -y nodejs >> /tmp/node_setup.log 2>&1
fi
node -v; npm -v
cd /content
wget -q "$KIT_URL" -O kit.tar.gz
rm -rf mona && mkdir mona && tar xzf kit.tar.gz -C mona
echo "--- kit contents ---"; ls mona; du -sh kit.tar.gz

In [ ]:
%%bash
# [SETUP 2/2] npm install chạy nền — poll bằng cell dưới
cd /content/mona
nohup npm install --no-audit --no-fund > /tmp/npm.log 2>&1 &
echo "npm install started (background)"

In [ ]:
%%bash
# [POLL] trạng thái npm install
if pgrep -f "npm install" > /dev/null; then echo "STATUS: installing..."; else echo "STATUS: DONE"; fi
tail -2 /tmp/npm.log || true
ls /content/mona/node_modules 2>/dev/null | wc -l

In [ ]:
%%bash
# [RENDER] single-process (đã benchmark: nhanh hơn chia chunk trên 1 máy)
# concurrency ĐỘNG theo core + x264 veryfast (CRF 18 giữ chất lượng)
cd /content/mona
CORES=$(nproc)
CONC=$(( CORES > 4 ? CORES - 2 : CORES ))
echo "runtime has $CORES cores -> concurrency=$CONC"
nohup npx remotion render src/index.tsx MonaTimeline out/mona_FINAL_1080_colab.mp4 \
  --props=public/mona_timeline_props.json --frames=0-2800 --crf=18 \
  --concurrency=$CONC --x264-preset=veryfast \
  > /tmp/render.log 2>&1 &
echo "render started $(date +%H:%M:%S)"

In [ ]:
%%bash
# [POLL] tiến độ render (an toàn khi log chưa có dòng Rendered)
date +%H:%M:%S
grep -oE "(Bundling|Rendered|Encoded) [0-9]+[%/0-9]*" /tmp/render.log | tail -2 || true
tail -1 /tmp/render.log || true
ls -la /content/mona/out/ 2>/dev/null || echo "(chưa có output)"

In [ ]:
%%bash
# [WAIT] chờ render xong rồi báo kết quả
while pgrep -f "remotion render" > /dev/null; do sleep 5; done
echo "=== RENDER FINISHED $(date +%H:%M:%S) ==="
tail -3 /tmp/render.log || true
ls -la /content/mona/out/

In [ ]:
%%bash
# [EXPORT] upload MP4 kết quả -> link tự hủy 1h để tải về máy
# (tải về local: curl -sL -o out/mona_FINAL_1080_colab.mp4 "<link>")
cd /content/mona/out
curl -s -F "reqtype=fileupload" -F "time=1h" -F "fileToUpload=@mona_FINAL_1080_colab.mp4" \
  "https://litterbox.catbox.moe/resources/internals/api.php"